# Bag of Words (BOW)

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.DataFrame({
    'text': ['people watch saylaniIT', 'saylaniIT watch people', 'people write comment', 'saylaniIT write comment'],
    'output': [1,1,0,0]
})

In [3]:
df

,text,output
0,people watch saylaniIT,1
1,saylaniIT watch people,1
2,people write comment,0
3,saylaniIT write comment,0


In [4]:
from sklearn.feature_extraction.text import CountVectorizer

In [5]:
cv = CountVectorizer()

In [6]:
bow = cv.fit_transform(df['text'])

In [7]:
bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 12 stored elements and shape (4, 5)>

In [8]:
cv.vocabulary_

{'people': 1, 'watch': 3, 'saylaniit': 2, 'write': 4, 'comment': 0}

In [9]:
print(bow[0].toarray())

[[0 1 1 1 0]]


In [10]:
print(bow[1].toarray())

[[0 1 1 1 0]]


# TF-IDF

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
tfidf = TfidfVectorizer()

In [13]:
tfidf.fit_transform(df['text']).toarray()

array([[0.        , 0.53256952, 0.53256952, 0.65782931, 0.        ],
       [0.        , 0.53256952, 0.53256952, 0.65782931, 0.        ],
       [0.61366674, 0.49681612, 0.        , 0.        , 0.61366674],
       [0.61366674, 0.        , 0.49681612, 0.        , 0.61366674]])

In [14]:
print(tfidf.idf_)

[1.51082562 1.22314355 1.22314355 1.51082562 1.51082562]


In [15]:
print(tfidf.get_feature_names_out())

['comment' 'people' 'saylaniit' 'watch' 'write']


________________________________________________________________________

In [17]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [18]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [19]:
data = {
    "review": [
        "I absolutely loved this movie! It was fantastic and thrilling.",
        "The movie was boring and too long. I did not enjoy it at all.",
        "Amazing acting, great story, and wonderful direction!",
        "Terrible film. Waste of time and money."
    ]
}

In [20]:
df = pd.DataFrame(data)

In [21]:
def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    return text

In [22]:
df['clean_text'] = df['review'].apply(clean_text)

In [23]:
stop_words = stopwords.words('english')

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in stop_words])


In [24]:
df['no_stopwords'] = df['clean_text'].apply(remove_stopwords)

In [25]:
# Tokenize
df['tokens'] = df['no_stopwords'].apply(word_tokenize)

In [26]:
# stemming

stemmer = PorterStemmer()

def stem_words(tokens):
    return [stemmer.stem(word) for word in tokens]

df['stemmed'] = df['tokens'].apply(stem_words)

# Join back for vectorization
df['stemmed_text'] = df['stemmed'].apply(lambda x: " ".join(x))

In [27]:
df

,review,clean_text,no_stopwords,tokens,stemmed,stemmed_text
0,I absolutely loved this movie! It was fantasti...,i absolutely loved this movie it was fantastic...,absolutely loved movie fantastic thrilling,"[absolutely, loved, movie, fantastic, thrilling]","[absolut, love, movi, fantast, thrill]",absolut love movi fantast thrill
1,The movie was boring and too long. I did not e...,the movie was boring and too long i did not en...,movie boring long enjoy,"[movie, boring, long, enjoy]","[movi, bore, long, enjoy]",movi bore long enjoy
2,"Amazing acting, great story, and wonderful dir...",amazing acting great story and wonderful direc...,amazing acting great story wonderful direction,"[amazing, acting, great, story, wonderful, dir...","[amaz, act, great, stori, wonder, direct]",amaz act great stori wonder direct
3,Terrible film. Waste of time and money.,terrible film waste of time and money,terrible film waste time money,"[terrible, film, waste, time, money]","[terribl, film, wast, time, money]",terribl film wast time money


In [28]:
# Lemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['lemmatized'] = df['tokens'].apply(lemmatize_words)

# Join back
df['lemmatized_text'] = df['lemmatized'].apply(lambda x: " ".join(x))

In [29]:
# Vectorizer

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(df['lemmatized_text'])

bow_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

In [30]:
bow_df

,absolutely,acting,amazing,boring,direction,enjoy,fantastic,film,great,long,loved,money,movie,story,terrible,thrilling,time,waste,wonderful
0,1,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0
1,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0
2,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1
3,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,1,1,0


In [34]:
print("=== Original Reviews ===")
print(df['review'])
print("\n=== After Preprocessing (Lemmatized) ===")
print(df['lemmatized_text'])

=== Original Reviews ===
0    I absolutely loved this movie! It was fantasti...
1    The movie was boring and too long. I did not e...
2    Amazing acting, great story, and wonderful dir...
3              Terrible film. Waste of time and money.
Name: review, dtype: object

=== After Preprocessing (Lemmatized) ===
0        absolutely loved movie fantastic thrilling
1                           movie boring long enjoy
2    amazing acting great story wonderful direction
3                    terrible film waste time money
Name: lemmatized_text, dtype: object


In [35]:
# Baf of words representation
bow_df

,absolutely,acting,amazing,boring,direction,enjoy,fantastic,film,great,long,loved,money,movie,story,terrible,thrilling,time,waste,wonderful
0,1,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0
1,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0
2,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1
3,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,1,1,0


In [37]:
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df['lemmatized_text'])

In [38]:
tfidf_df = pd.DataFrame(X.toarray(), columns=tfidf.get_feature_names_out())

In [39]:
tfidf_df

,absolutely,acting,amazing,boring,direction,enjoy,fantastic,film,great,long,loved,money,movie,story,terrible,thrilling,time,waste,wonderful
0,0.465162,0.000000,0.000000,0.000000,0.000000,0.000000,0.465162,0.000000,0.000000,0.000000,0.465162,0.000000,0.366739,0.000000,0.000000,0.465162,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.525473,0.000000,0.525473,0.000000,0.000000,0.000000,0.525473,0.000000,0.000000,0.414289,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.408248,0.408248,0.000000,0.408248,0.000000,0.000000,0.000000,0.408248,0.000000,0.000000,0.000000,0.000000,0.408248,0.000000,0.000000,0.000000,0.000000,0.408248
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.447214,0.000000,0.000000,0.000000,0.447214,0.000000,0.000000,0.447214,0.000000,0.447214,0.447214,0.000000
